# Qwen3-Reranker-8B (cross-encoder) на A100

Реранкинг top-20 документов из 4 input-runs через **Qwen/Qwen3-Reranker-8B** — специализированный cross-encoder для IR.

**Архитектура.** В отличие от listwise LLM-реранкинга (DeepSeek), здесь модель делает forward pass на каждую пару `(query, doc)` и выдаёт **logp(yes) / logp(no)** для специального инструктивного промпта. Скор релевантности = `softmax([logp(yes), logp(no)])[0]`. Это интерфейс из карточки модели на HuggingFace.

**Input runs:** четыре файла `runs/`:
* `rrf_query2doc_rus-scifact.json`, `rrf_query2doc_rus-nfcorpus.json`
* `rrf_query2doc_csqe_rus-scifact.json`, `rrf_query2doc_csqe_rus-nfcorpus.json`

**Output:** 4 файла `runs/rerank_qwen3_*.json` + сводная таблица NDCG/Recall/MAP @ 5,10.

**Hardware:** A100 80GB. Модель в bf16, flash-attention 2, top-20 реранкинг → ~2-3 минуты на оба датасета.

**Объём вычислений:** 623 запроса × 4 input-runs × 20 документов = ~50 тыс forward pass. Кэш по hash(query, doc_ids) убирает дубликаты между q2d и q2d_csqe (там часто пересекается top-20).

## 0. Установка зависимостей

In [ ]:
!pip install -q datasets 'transformers>=4.51.0' accelerate pytrec_eval tqdm pandas einops

In [ ]:
from __future__ import annotations

import os
import sys
import json
import time
import hashlib
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import pytrec_eval

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)
else:
    print('WARNING: GPU не найден. Этот ноутбук рассчитан на A100 80GB.')

# === Директории ===
ROOT = Path('.')
RUN_DIR = ROOT / 'runs'
RES_DIR = ROOT / 'results'
CACHE_DIR = ROOT / 'cache'
RERANK_CACHE = CACHE_DIR / 'rerank_qwen3'
for d in (RUN_DIR, RES_DIR, CACHE_DIR, RERANK_CACHE):
    d.mkdir(parents=True, exist_ok=True)

# === Конфигурация реранкинга ===
QWEN_MODEL_ID = 'Qwen/Qwen3-Reranker-8B'
RERANK_TOP_K = 20             # сколько документов реранкить
RERANK_DOC_TRUNCATE = 1500    # символов на документ перед токенизацией
MAX_LENGTH = 512              # max токенов на пару (query + instruction + doc)
BATCH_SIZE = 16               # пар на batch
DTYPE = torch.bfloat16

## 1. Положи входные runs в ./runs/

In [ ]:
INPUT_CONFIGS = [
    ('q2d',      'rus-scifact',  RUN_DIR / 'rrf_query2doc_rus-scifact.json'),
    ('q2d',      'rus-nfcorpus', RUN_DIR / 'rrf_query2doc_rus-nfcorpus.json'),
    ('q2d_csqe', 'rus-scifact',  RUN_DIR / 'rrf_query2doc_csqe_rus-scifact.json'),
    ('q2d_csqe', 'rus-nfcorpus', RUN_DIR / 'rrf_query2doc_csqe_rus-nfcorpus.json'),
]

missing = [str(p) for _, _, p in INPUT_CONFIGS if not p.exists()]
if missing:
    print('ОШИБКА: отсутствуют входные runs:')
    for m in missing:
        print(f'  - {m}')
    sys.exit(1)
else:
    print('Все 4 входных run-файла на месте.')

## 2. Загрузка датасетов (корпус + queries + qrels)

In [ ]:
def load_beir_like(name: str, qrels_split: str = 'test'):
    corpus_ds = load_dataset(f'kaengreg/{name}', 'corpus')['train']
    queries_ds = load_dataset(f'kaengreg/{name}', 'queries')['train']
    qrels_ds = load_dataset(f'kaengreg/{name}-qrels', 'qrels')[qrels_split]

    corpus = {}
    for row in corpus_ds:
        corpus[str(row['_id'])] = {
            'title': row.get('title') or '',
            'text': row.get('text') or '',
        }

    qrels = defaultdict(dict)
    for row in qrels_ds:
        qrels[str(row['query-id'])][str(row['corpus-id'])] = int(row['score'])

    valid_qids = set(qrels.keys())
    queries = {}
    for row in queries_ds:
        qid = str(row['_id'])
        if qid in valid_qids:
            queries[qid] = {'text': row.get('text') or ''}
    qrels = {q: r for q, r in qrels.items() if q in queries}

    print(f'  {name}: corpus={len(corpus)}, queries={len(queries)}, qrels={len(qrels)}')
    return corpus, queries, dict(qrels)


DATASET_NAMES = ['rus-scifact', 'rus-nfcorpus']
DATASETS = {ds: load_beir_like(ds, 'test') for ds in DATASET_NAMES}

## 3. Метрики (NDCG/Recall/MAP @5 и @10)

In [ ]:
K_VALUES = [5, 10]


def evaluate_run(qrels: dict, run: dict, k_values=K_VALUES):
    metric_strs = set()
    for k in k_values:
        metric_strs.add(f'ndcg_cut.{k}')
        metric_strs.add(f'recall.{k}')
        metric_strs.add(f'map_cut.{k}')
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metric_strs)
    per_q = evaluator.evaluate(run)
    agg = defaultdict(list)
    for q, m in per_q.items():
        for k, v in m.items():
            agg[k].append(v)
    means = {k: float(np.mean(v)) for k, v in agg.items()}
    out = {}
    for k in k_values:
        out[f'NDCG@{k}'] = means.get(f'ndcg_cut_{k}', 0.0)
        out[f'Recall@{k}'] = means.get(f'recall_{k}', 0.0)
        out[f'MAP@{k}'] = means.get(f'map_cut_{k}', 0.0)
    return out

## 4. Загрузка Qwen3-Reranker-8B

Грузим один раз, держим в bf16 на GPU. Используем `flash_attention_2` если установлен, иначе fallback на eager.

In [ ]:
print(f'Loading {QWEN_MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, padding_side='left')

load_kwargs = dict(torch_dtype=DTYPE)
try:
    model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID, attn_implementation='flash_attention_2', **load_kwargs
    ).to(DEVICE)
    print('  using flash_attention_2')
except (ImportError, ValueError) as e:
    print(f'  flash_attention_2 unavailable ({type(e).__name__}); falling back to eager')
    model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL_ID, **load_kwargs).to(DEVICE)
model.eval()

# Идентификаторы токенов yes/no — нужны для извлечения скора
TOKEN_YES = tokenizer('yes', add_special_tokens=False)['input_ids'][0]
TOKEN_NO = tokenizer('no', add_special_tokens=False)['input_ids'][0]
print(f'  token id for "yes": {TOKEN_YES}, for "no": {TOKEN_NO}')
print(f'  VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 5. Промпт-формат и форвард-функция

Промпт-формат из карточки модели: системная инструкция, затем `<Instruct>: ...` (опциональная задача), `<Query>: ...`, `<Document>: ...`. Модель должна предсказать токен `yes` или `no`. Скор = `softmax([logits_yes, logits_no])[0]` — вероятность класса релевантного.

Используется chat-template Qwen с `enable_thinking=False`, чтобы модель сразу выдавала ответ без CoT (как в их официальном примере).

In [ ]:
RERANK_INSTRUCTION = (
    'Given a web search query, retrieve relevant passages that answer the query'
)
PREFIX_TOKENS = (
    '<|im_start|>system\nJudge whether the Document meets the requirements based on '
    'the Query and the Instruct provided. Note that the answer can only be "yes" or '
    '"no".<|im_end|>\n<|im_start|>user\n'
)
SUFFIX_TOKENS = (
    '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
)
PREFIX_IDS = tokenizer(PREFIX_TOKENS, add_special_tokens=False)['input_ids']
SUFFIX_IDS = tokenizer(SUFFIX_TOKENS, add_special_tokens=False)['input_ids']


def _format_pair(query: str, doc: str) -> str:
    """Построение пользовательского сообщения (без префикса/суффикса)."""
    return (f'<Instruct>: {RERANK_INSTRUCTION}\n'
            f'<Query>: {query}\n'
            f'<Document>: {doc}')


@torch.inference_mode()
def score_pairs(pairs: list[tuple[str, str]], batch_size: int = BATCH_SIZE,
                max_length: int = MAX_LENGTH) -> np.ndarray:
    """pairs: list of (query, doc) → np.ndarray of relevance scores in [0, 1].
    Скор = softmax([logit_yes, logit_no])[0] на последнем токене.
    """
    if not pairs:
        return np.array([])
    scores: list[float] = []
    for s in tqdm(range(0, len(pairs), batch_size), desc='qwen3 score', leave=False):
        batch = pairs[s:s + batch_size]
        # Токенизируем середину (без prefix/suffix), затем приклеиваем
        middle = [_format_pair(q, d) for q, d in batch]
        max_middle = max_length - len(PREFIX_IDS) - len(SUFFIX_IDS)
        enc = tokenizer(middle, add_special_tokens=False,
                        truncation=True, max_length=max_middle,
                        return_attention_mask=False)
        input_ids_list = []
        for ids in enc['input_ids']:
            input_ids_list.append(PREFIX_IDS + ids + SUFFIX_IDS)
        # Pad слева (генерация после последнего токена), -> attention mask
        max_len = max(len(x) for x in input_ids_list)
        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        padded_ids, attn_masks = [], []
        for ids in input_ids_list:
            n_pad = max_len - len(ids)
            padded_ids.append([pad_id] * n_pad + ids)
            attn_masks.append([0] * n_pad + [1] * len(ids))
        input_ids = torch.tensor(padded_ids, dtype=torch.long, device=DEVICE)
        attention_mask = torch.tensor(attn_masks, dtype=torch.long, device=DEVICE)
        # Forward (только последний токен нужен)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out.logits[:, -1, :]                    # (B, V)
        yes_no = logits[:, [TOKEN_YES, TOKEN_NO]]        # (B, 2)
        probs = torch.softmax(yes_no.float(), dim=-1)
        scores.extend(probs[:, 0].cpu().numpy().tolist())
    return np.array(scores, dtype=np.float32)

## 6. Реранкинг одного run-а

Для каждого qid:
1. Берём top-20 doc_id из исходного run'а.
2. Формируем 20 пар `(query, doc_text)`.
3. Скорим все пары пакетом.
4. Сортируем по убыванию скора.
5. Документы с позиций 21+ из исходного run'а **отбрасываются**.

Кэш — по `hash(query + doc_ids)`. Если в q2d и q2d_csqe top-20 совпадает по содержанию и порядку, делается **один проход** через GPU.

In [ ]:
def _cache_key(query: str, doc_ids: list[str]) -> str:
    h = hashlib.sha256()
    h.update(query.encode('utf-8'))
    h.update(b'|')
    h.update('|'.join(doc_ids).encode('utf-8'))
    return h.hexdigest()[:16]


def _doc_text(corpus: dict, did: str, truncate: int = RERANK_DOC_TRUNCATE) -> str:
    d = corpus.get(did, {})
    return ((d.get('title', '') + ' ' + d.get('text', '')).strip())[:truncate]


def rerank_run(run: dict, queries: dict, corpus: dict,
               top_k: int = RERANK_TOP_K) -> dict:
    """Возвращает run с переупорядоченным top-K (позиции 21+ отброшены)."""
    # 1) Готовим пары для всех qid с учётом кэша
    qid_to_doc_ids: dict[str, list[str]] = {}
    qid_to_cache: dict[str, str] = {}
    pairs: list[tuple[str, str]] = []
    pair_route: list[str] = []  # для каждой пары — qid
    cached: dict[str, np.ndarray] = {}  # cache_key -> scores

    for qid, ranked in run.items():
        if qid not in queries:
            qid_to_doc_ids[qid] = []
            continue
        sorted_docs = sorted(ranked.items(), key=lambda x: -x[1])[:top_k]
        doc_ids = [d for d, _ in sorted_docs]
        qid_to_doc_ids[qid] = doc_ids
        if not doc_ids:
            continue

        query_text = queries[qid]['text']
        ck = _cache_key(query_text, doc_ids)
        qid_to_cache[qid] = ck

        # Проверяем кэш
        cache_path = RERANK_CACHE / f'{ck}.json'
        if cache_path.exists():
            cached[ck] = np.array(
                json.loads(cache_path.read_text(encoding='utf-8'))['scores'],
                dtype=np.float32,
            )
            continue
        if ck in cached:  # уже добавили в текущем прогоне
            continue

        # Иначе — ставим в очередь на скоринг
        for did in doc_ids:
            pairs.append((query_text, _doc_text(corpus, did)))
            pair_route.append(qid)

    # 2) Скоринг свежих пар
    if pairs:
        print(f'    GPU scoring: {len(pairs):,} pairs')
        scores = score_pairs(pairs, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
        # Группируем обратно по qid — ВАЖНО: в том же порядке, в котором добавлялись пары
        offset = 0
        seen_qids: list[str] = []  # упорядоченный список уникальных qid в pair_route
        seen_set: set[str] = set()
        for qid in pair_route:
            if qid not in seen_set:
                seen_set.add(qid)
                seen_qids.append(qid)
        for qid in seen_qids:
            ck = qid_to_cache[qid]
            n = len(qid_to_doc_ids[qid])
            qid_scores = scores[offset:offset + n]
            offset += n
            cached[ck] = qid_scores
            # Записать в кэш на диск
            (RERANK_CACHE / f'{ck}.json').write_text(
                json.dumps({'scores': qid_scores.tolist()}, ensure_ascii=False),
                encoding='utf-8',
            )
    else:
        print('    All pairs cached.')

    # 3) Собираем новый run
    new_run: dict[str, dict[str, float]] = {}
    for qid in run.keys():
        doc_ids = qid_to_doc_ids.get(qid, [])
        if not doc_ids:
            new_run[qid] = {}
            continue
        ck = qid_to_cache[qid]
        qid_scores = cached[ck]
        new_run[qid] = {did: float(s) for did, s in zip(doc_ids, qid_scores)}
    return new_run

## 7. Прогон по 4 input-runs

In [ ]:
all_metrics = []

for cfg_name, ds_name, run_path in INPUT_CONFIGS:
    corpus, queries, qrels = DATASETS[ds_name]
    print(f'\n{"=" * 70}\n{cfg_name}  on  {ds_name}\n{"=" * 70}')

    src_run = json.loads(run_path.read_text(encoding='utf-8'))
    print(f'  Loaded {len(src_run)} queries from {run_path.name}')

    # Бейзлайн (без реранкинга)
    base_m = evaluate_run(qrels, src_run)
    all_metrics.append({
        'dataset': ds_name, 'retrieval': cfg_name, 'rerank': 'none',
        **base_m,
    })
    print(f'  baseline: NDCG@10={base_m["NDCG@10"]:.4f}  '
          f'Recall@10={base_m["Recall@10"]:.4f}  MAP@10={base_m["MAP@10"]:.4f}')

    # Реранкинг
    t0 = time.time()
    new_run = rerank_run(src_run, queries, corpus, top_k=RERANK_TOP_K)
    print(f'  rerank done in {time.time() - t0:.1f}s')

    out_path = RUN_DIR / f'rerank_qwen3_{cfg_name}_{ds_name}.json'
    out_path.write_text(json.dumps(new_run, ensure_ascii=False), encoding='utf-8')
    print(f'  saved {out_path}')

    rerank_m = evaluate_run(qrels, new_run)
    all_metrics.append({
        'dataset': ds_name, 'retrieval': cfg_name, 'rerank': 'qwen3',
        **rerank_m,
    })
    print(f'  + Qwen3 rerank: NDCG@10={rerank_m["NDCG@10"]:.4f}  '
          f'Recall@10={rerank_m["Recall@10"]:.4f}  MAP@10={rerank_m["MAP@10"]:.4f}')

## 8. Сводная таблица

In [ ]:
df = pd.DataFrame(all_metrics)
metric_cols = ['NDCG@5', 'NDCG@10', 'Recall@5', 'Recall@10', 'MAP@5', 'MAP@10']
for col in metric_cols:
    df[col] = df[col].round(4)
df = df[['dataset', 'retrieval', 'rerank'] + metric_cols]
df.sort_values(['dataset', 'retrieval', 'rerank'], inplace=True)
df.to_csv(RES_DIR / 'rerank_qwen3_summary.csv', index=False)
df

In [ ]:
# Wide-формат: дельты от реранкинга
rows = []
for (ds_name, ret_name), grp in df.groupby(['dataset', 'retrieval']):
    g = grp.set_index('rerank')
    if 'none' in g.index and 'qwen3' in g.index:
        for col in metric_cols:
            rows.append({
                'dataset': ds_name,
                'retrieval': ret_name,
                'metric': col,
                'no rerank': g.loc['none', col],
                'qwen3 rerank': g.loc['qwen3', col],
                'delta': round(g.loc['qwen3', col] - g.loc['none', col], 4),
            })
delta_df = pd.DataFrame(rows)
delta_df.to_csv(RES_DIR / 'rerank_qwen3_deltas.csv', index=False)
delta_df